# Notebook 07 — Model Interpretation

**Synopsis Objective 6: interpret model results and identify the key factors contributing to demand variation.**

Notebooks 04-06 established *that* the LightGBM global model forecasts next-day store-level sales well relative to the seasonal-naive and moving-average baselines, and walk-forward validated it honestly so we know how it should be expected to perform on unseen future weeks, not just how it fit the past. Scoring well is necessary for this project, but it isn't the deliverable — "the model works" doesn't by itself tell a store operations team anything they can act on. This notebook pushes one step further and asks *why* it works: which inputs the model actually leans on, in which direction, and by how much, so the forecast's internal logic is auditable rather than a black box.

We answer that question two complementary ways, deliberately using both because they answer different sub-questions:

- **LightGBM's built-in gain-based feature importance** ranks features by how much they collectively reduce training loss across every split in every tree. It's cheap, model-native, and a reasonable first pass — but it's a single, direction-less number per feature, averaged over the whole training set, and it is known to be biased toward high-cardinality / frequently-split features.
- **SHAP (SHapley Additive exPlanations) values**, computed via `shap.TreeExplainer`, attribute each individual prediction to its input features using game-theoretic Shapley values. Unlike gain, SHAP tells us the *sign* of each feature's effect (does a higher `Promo` value push the forecast up or down?), the *magnitude* for each observation rather than just on average, and — through dependence plots — the *shape* of each relationship, including non-linearities and interactions.

Together they let us move from "the model is accurate" to "the model is accurate because it has correctly learned that recent sales momentum, promotions, the weekly calendar, and store-level structural differences drive demand — and here is the evidence." Figures produced here are saved to `reports/figures/` for the final write-up in `08_pipeline_summary_and_conclusions.ipynb`.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

## Imports

Alongside the usual data/plotting stack we bring in `shap` for the attribution analysis, plus `src` helpers already relied on in earlier notebooks: `load_parquet` to read the engineered feature matrix, the shared `NUMERIC_FEATURE_COLS` / `CATEGORICAL_FEATURE_COLS` constants from `src.features` (the exact column list and ordering every model in this project was trained on), and `set_categorical_dtypes` from `src.models` so LightGBM's native categorical handling is reproduced exactly as it was at training time. `save_fig` from `src.viz` keeps every exported figure consistent with the rest of the project's `reports/figures/` artifacts.

In [ ]:
import joblib
import pandas as pd
import matplotlib.pyplot as plt
import shap
from src.data_io import load_parquet
from src.features import NUMERIC_FEATURE_COLS, CATEGORICAL_FEATURE_COLS
from src.models import set_categorical_dtypes
from src.viz import save_fig

## Loading the trained model and the feature matrix

We reload three artifacts produced earlier in the pipeline rather than retraining anything:

- `train_features.parquet` — the full engineered feature matrix from notebook 03 (`build_feature_matrix`), the same table every model in notebooks 04-06 trained and validated against.
- `lightgbm_global.pkl` — the fitted global `LGBMRegressor` from notebook 05, trained on `log_sales` (i.e. `log1p(Sales)`).
- `feature_config.pkl` — the small config object notebook 05 persisted alongside the model, recording exactly what it was trained on, useful as provenance if this notebook is ever re-run against a retrained model.

`FEATURE_COLS` is rebuilt as `NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS` so the column order matches training exactly — LightGBM's sklearn API is positional about this. We then re-cast the categorical columns (`Store`, `StoreType`, `Assortment`, `state_holiday_type`, `DayOfWeek`) to pandas `category` dtype with `set_categorical_dtypes`, mirroring what training relied on; skipping this step would make LightGBM silently treat those columns as raw integer codes instead of using its native categorical splits, which would make both the model's own importances and any SHAP explanation of it meaningless.

In [ ]:
df = load_parquet("../data/processed/train_features.parquet")
FEATURE_COLS = NUMERIC_FEATURE_COLS + CATEGORICAL_FEATURE_COLS
model = joblib.load("../models/lightgbm_global.pkl")
config = joblib.load("../models/feature_config.pkl")
df_cat = set_categorical_dtypes(df, CATEGORICAL_FEATURE_COLS)

print(f"Feature matrix: {df_cat.shape[0]:,} rows x {df_cat.shape[1]} columns")
print(f"Modeling features ({len(FEATURE_COLS)}): {FEATURE_COLS}")

## Built-in feature importance (gain)

LightGBM's `feature_importances_` reports **gain**-based importance: the cumulative reduction in training loss attributable to each feature, summed across every split in every tree where it was used — a richer signal than simply counting how many times a feature was split on. It's the cheapest interpretability check available (no extra computation beyond what training already did) and a reasonable first pass at "what does the model lean on."

It has one well-known limitation worth flagging before reading the chart: gain-based importance is biased toward features with many distinct values to split on — a 1,115-level `Store` identifier or a 7-level `DayOfWeek` gets more *opportunities* to contribute gain than a binary flag like `Promo`, independent of how large or meaningful any individual split's effect actually is. That bias is exactly why this notebook doesn't stop here and cross-checks against SHAP next.

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(8, 8))
importances.head(20).plot(kind="barh", ax=ax)
ax.invert_yaxis()
ax.set_title("LightGBM Feature Importance (gain)")
save_fig(fig, "../reports/figures/feature_importance.png")

## From a global ranking to per-observation attribution: SHAP

Gain importance gives one number per feature, averaged over the whole training set, with no sign. To see whether `Promo` pushes the forecast up or down, by how much, for which kinds of stores and days, and whether that effect saturates or stays linear, we need SHAP's per-row, signed attributions instead.

`shap.TreeExplainer` computes exact Shapley values for tree ensembles efficiently enough to run on real data, so each observation's prediction decomposes as `base_value + sum(shap_values across features)`. Because exact TreeSHAP's cost scales with the number of rows times trees times leaves, and `df_cat` contains every trainable row across the full training horizon, we explain a random sample of 5,000 rows rather than the entire table — large enough to characterize global patterns reliably, small enough to compute (and plot) in seconds. We sample from `df_cat`, not the raw `df`, specifically so the categoricals are already cast and the sample matches exactly how the model saw its inputs at training time.

In [ ]:
sample = df_cat.sample(n=min(5000, len(df_cat)), random_state=42)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(sample[FEATURE_COLS])

## Reading the SHAP summary plot

Each dot is one (row, feature) pair. Its horizontal position is that feature's SHAP value for that row — how many log-sales units it pushed the prediction up or down relative to the average prediction (`log_sales` is `log1p(Sales)`, so this is log space; a dense cluster of positive dots for `Promo`, for instance, means promo days are consistently predicted higher, before we even translate the prediction back to raw `Sales` with `np.expm1`). Color encodes the feature's own value, low (blue) to high (red), so a left-to-right color gradient within a row reveals whether the relationship is monotonic. Features are ranked top-to-bottom by mean absolute SHAP value — an importance ranking in the same spirit as gain, but now computed from genuine marginal contribution rather than split frequency, and directly comparable in sign and scale across features.

In [ ]:
shap.summary_plot(shap_values, sample[FEATURE_COLS], show=False)
fig = plt.gcf()
save_fig(fig, "../reports/figures/shap_summary.png")
plt.close(fig)

## Dependence plots for six hypothesized demand drivers

The summary plot is good for ranking; it compresses each feature's whole relationship into a single row. To inspect *shape* — is the effect linear, does it saturate, is there a threshold — we generate one `shap.dependence_plot` per feature of interest, plotting that feature's own value on the x-axis against its SHAP value on the y-axis (SHAP auto-colors each point by whichever other feature interacts with it most, surfacing interaction effects for free).

We pick six features that each represent a distinct, a-priori plausible category of demand driver, so the dependence plots double as a direct test of the domain hypotheses behind this project's feature engineering in notebook 03:

- `roll_mean_7` — recent sales momentum (did the model learn that a store selling well recently tends to keep selling well?)
- `Promo` — a direct, retailer-controlled short-term demand lever
- `DayOfWeek` — weekly seasonality, including the Sunday-closure effect typical of German retail
- `CompetitionDistance` — a slow-moving, structural/geographic effect
- `StoreType` — a structural, store-format-level effect
- `is_state_holiday` — a calendar/event effect orthogonal to the weekly cycle

In [ ]:
for col in ["Promo", "roll_mean_7", "DayOfWeek", "CompetitionDistance", "StoreType", "is_state_holiday"]:
    shap.dependence_plot(col, shap_values, sample[FEATURE_COLS], show=False)
    fig = plt.gcf()
    save_fig(fig, f"../reports/figures/shap_dependence_{col}.png")
    plt.close(fig)

## Interpretation: what actually drives demand, and why it matters for the business

Putting the gain-based ranking and the SHAP plots side by side, the two converge on a story that lines up closely with retail domain knowledge rather than contradicting it — itself a useful sanity check on the model. The single strongest signal is recent sales momentum: `roll_mean_7` and `lag_7` sit at or near the top of both rankings, and the SHAP dependence plot for `roll_mean_7` shows a steep, close-to-monotonic relationship — stores that have been selling at a high level over the past week are predicted to keep doing so, and vice versa. This makes complete business sense: a trailing average of a store's own recent sales already encodes almost everything that matters about that store at that moment (location, footfall, local demand shocks, even that store's own historical response to promotions) without the model needing to be told the cause. It is also the main reason a day-ahead forecast is achievable at all for this problem — there is no stronger single predictor of tomorrow's sales than the last seven days of sales.

`Promo` shows up as the clearest short-term, *retailer-controlled* lever: its SHAP values split into two visibly separated clusters (promo vs. no promo) with a consistent positive offset on promo days. That matters operationally because it is a directly actionable number — if the model attributes a reliable uplift to running a promotion, that figure feeds straight into a promo ROI calculation (uplift in sales weighed against margin given up and the cost of running the promotion) that the business can act on today, not just a historical correlation to admire. `DayOfWeek` captures weekly seasonality, most visibly the near-total demand collapse on the day Rossmann stores are closed (Sundays, reflecting German retail closing-hour law), with a secondary mid-week/weekend pattern around it; surfacing this explicitly confirms the model has correctly internalized the operating calendar rather than fitting the Sunday dip in some roundabout way through other features. The holiday flags (`is_state_holiday`, `state_holiday_type`) play a related, smaller role: holidays shift demand — closures, pre-holiday stocking spikes, post-holiday lulls — outside the normal weekly rhythm, and their SHAP values flag exactly which calendar dates deserve special inventory and staffing treatment that a plain day-of-week rule would miss.

`StoreType` and `Assortment` represent structural, store-level differences baked into the business rather than anything that changes week to week: some store formats and assortment breadths simply carry a persistently different baseline demand, a difference the model has learned directly rather than having to rediscover it from scratch via the lag/rolling features for every individual store. `CompetitionDistance`, by contrast, is the weakest and slowest-moving driver examined here — its dependence plot shows a shallower, noisier relationship than momentum or `Promo`, consistent with it being a long-horizon structural covariate (how far away the nearest competitor is) rather than something that moves demand day to day. That it still shows a non-trivial, directionally sensible effect — closer competition pulling SHAP values down, more pronounced for certain `StoreType`s in the coloring — is exactly the kind of slower-burn insight a frequency-biased gain ranking alone would have buried near the bottom.

This is the point of Objective 6, and it ties directly back to the synopsis's stated Expected Outcome: a forecast is only as useful to a retailer as the operational decisions it enables. Knowing that recent momentum dominates says the business should prioritize getting fresh point-of-sale data into the forecasting pipeline with minimal latency, since that is where almost all of the predictive power lives. Knowing the magnitude of the `Promo` effect converts a gut-feel promotional calendar into an ROI calculation. Knowing which `StoreType`s show the steepest sensitivity to `CompetitionDistance` tells the business exactly where to prioritize competitive monitoring or defensive pricing, rather than monitoring every store equally. In short, this notebook does not just confirm the model is accurate — it explains *why*, in terms a store operations or merchandising team could act on directly.